<a href="https://colab.research.google.com/github/fserracrespi/ir2_Project/blob/mhm-dev/pairs/10-gpl/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPL demo: zero-shot vs GenQ vs GPL, side by side

Type any query and see the top documents each model retrieves from the corpus. For test queries, the relevant documents from the qrels are marked ✓ (display only; nothing here trains on them).

Run all cells, then open the link that the last cell prints. The trained models are loaded from the folder they are saved in by `reproduce.ipynb`, and the document embeddings are cached on Drive after the first run, so later starts take seconds.

In [2]:
import os
if not os.path.exists("testbed.py"):
    if not os.path.exists("/content/ir2_Project"):
        !git clone -q -b mhm-dev https://github.com/fserracrespi/ir2_Project /content/ir2_Project
    %cd /content/ir2_Project
!pip install -q -r requirements.txt gradio
# from google.colab import drive
# drive.mount("/content/drive")

## Config

The demo shows one student on one dataset at a time. The folder names must match the runs whose models are saved.

In [ ]:
DATASET = "scifact"                       # "scifact" | "nfcorpus" | "arguana"
STUDENT_MODEL = "BAAI/bge-small-en-v1.5"  # or "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 5
# Colab without Drive: upload your files to /content/gpl
# Colab with Drive: uncomment drive.mount(...) and use "/content/drive/MyDrive/gpl"
# Locally: use "demo_data" or the full path to your folder
DRIVE = "/content/gpl"


# per-corpus settings, as in reproduce.ipynb
SETTINGS = {"scifact":  {"TRAIN_FRACTION": 1/2, "QUERIES_PER_DOC": 3},
            "nfcorpus": {"TRAIN_FRACTION": 1/2, "QUERIES_PER_DOC": 4},
            "arguana":  {"TRAIN_FRACTION": 1/3, "QUERIES_PER_DOC": 3}}
FRAC, QPP = SETTINGS[DATASET]["TRAIN_FRACTION"], SETTINGS[DATASET]["QUERIES_PER_DOC"]
STUD_SHORT = "bge" if "bge" in STUDENT_MODEL.lower() else "minilm"

GPL_FOLDER = f"gpl_{STUD_SHORT}_dense50_w200_frac{FRAC}_qpp{QPP}_neg1_rescale2_lr2e-05_seed0"
GENQ_FOLDER = f"genq_{STUD_SHORT}_w200_frac{FRAC}_qpp{QPP}_bs75_lr2e-05_seed0"
MODELS = {                                   # column title -> model name or saved folder
    "Zero-shot": STUDENT_MODEL,
    "GenQ":      f"{DRIVE}/models/{GENQ_FOLDER}/{DATASET}",
    "GPL":       f"{DRIVE}/models/{GPL_FOLDER}/{DATASET}",
}
for name, path in MODELS.items():
    ok = (not path.startswith("/")) or os.path.isdir(path)
    print(f"{name:10} {'ok     ' if ok else 'MISSING'} {path}")

## Load the corpus, the models and the document embeddings

In [ ]:
import json, html
import numpy as np
import testbed
from sentence_transformers import SentenceTransformer

corpus, queries, qrels = testbed.load(DATASET)
doc_ids = list(corpus)
doc_texts = testbed.doc_texts(corpus)          # the testbed's 200-word documents, as in the runs
QUERY_PREFIX = testbed.BGE_QUERY if STUD_SHORT == "bge" else ""
test_by_text = {t.strip().lower(): q for q, t in queries.items()}   # to recognise test queries
os.makedirs(f"{DRIVE}/demo_cache", exist_ok=True)

loaded = {}
for name, path in MODELS.items():
    model = SentenceTransformer(path, device=testbed.device())
    model.max_seq_length = 350
    cache = f"{DRIVE}/demo_cache/{DATASET}_{STUD_SHORT}_{name}.npy"
    if os.path.exists(cache):
        D = np.load(cache)
    else:
        D = model.encode(doc_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
        np.save(cache, D)
    loaded[name] = (model, D)
    print(name, D.shape)

In [ ]:
def search(query, k=TOP_K):
    # top-k documents per model, with cosine scores, exactly as testbed.dense_search ranks them
    out = {}
    for name, (model, D) in loaded.items():
        q = model.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0]
        s = D @ q
        top = np.argsort(-s)[:k + 1]
        out[name] = [(doc_ids[j], float(s[j])) for j in top]
    return out

def render(query):
    query = (query or "").strip()
    if not query:
        return "<p>Type a query.</p>"
    qid = test_by_text.get(query.lower())
    rel = qrels.get(qid, {}) if qid else {}
    results = search(query)
    head = (f"<p><b>Test query</b> <code>{html.escape(qid)}</code>: relevant documents are marked ✓ "
            f"({len(rel)} relevant in total).</p>" if qid else
            "<p><b>Free query</b>: not in the test set, so there are no relevance labels.</p>")
    cols = []
    for name, hits in results.items():
        if DATASET == "arguana" and qid:                       # as in the testbed: a query never retrieves itself
            hits = [(d, s) for d, s in hits if d != qid]
        hits = hits[:TOP_K]
        found = sum(1 for d, _ in hits if d in rel)
        items = []
        for rank, (d, s) in enumerate(hits, 1):
            mark = "✓ " if d in rel else ""
            title = html.escape(corpus[d]["title"] or "(no title)")
            snippet = html.escape(corpus[d]["text"][:220]) + "…"
            items.append(f"<li style='margin-bottom:10px'><b>{mark}{rank}. {title}</b> "
                         f"<span style='color:#666'>· score {s:.3f}</span><br>"
                         f"<span style='font-size:0.9em;color:#444'>{snippet}</span></li>")
        summary = f" · {found} relevant in top {TOP_K}" if qid else ""
        cols.append(f"<div style='flex:1;min-width:260px'><h3>{name}{summary}</h3>"
                    f"<ol style='list-style:none;padding-left:0'>{''.join(items)}</ol></div>")
    return head + "<div style='display:flex;gap:20px'>" + "".join(cols) + "</div>"

## Pick the three demo queries

**Hero:** a test query where GPL (or GenQ) does better than zero-shot. **Failure:** a test query where GPL loses the relevant document. **Wildcard:** a free query to type live. This cell suggests candidates from the saved runs. Useful to choose from.

In [ ]:
def per_query_ndcg(folder):
    run = json.load(open(f"{DRIVE}/runs/{folder}/{DATASET}.json"))
    qids, nd, _ = testbed.score_run(run, DATASET)
    return dict(zip(qids, nd))

base = testbed.stored()[DATASET]
zero = dict(zip(base["qids"], base[STUD_SHORT]["ndcg10"]))
gpl, genq = per_query_ndcg(GPL_FOLDER), per_query_ndcg(GENQ_FOLDER)

gain = sorted(queries, key=lambda q: gpl[q] - zero[q], reverse=True)
loss = sorted(queries, key=lambda q: gpl[q] - zero[q])
print("Hero candidates (GPL better than zero-shot):")
for q in gain[:5]:
    print(f"  zero {zero[q]:.2f}  GenQ {genq[q]:.2f}  GPL {gpl[q]:.2f} | {queries[q][:100]}")
print("\nFailure candidates (GPL loses most against zero-shot):")
for q in loss[:5]:
    print(f"  zero {zero[q]:.2f}  GenQ {genq[q]:.2f}  GPL {gpl[q]:.2f} | {queries[q][:100]}")

In [ ]:
# paste the queries you chose (the wildcard is only a suggestion: type your own live)
HERO = queries[gain[0]]
FAILURE = queries[loss[0]]
WILDCARD = "does coffee cause cancer"

## The demo

`share=True` prints a public link that works in any browser for 72 hours.

In [ ]:
import gradio as gr

with gr.Blocks(title="GPL demo") as demo:
    gr.Markdown(f"## GPL demo · {DATASET} · student {STUD_SHORT}\nZero-shot vs GenQ vs GPL, top {TOP_K} documents.")
    with gr.Row():
        box = gr.Textbox(label="Query", placeholder="Type a query and press Enter", scale=4)
        go = gr.Button("Search", variant="primary", scale=1)
    with gr.Row():
        b1, b2, b3 = gr.Button("Hero"), gr.Button("Failure"), gr.Button("Wildcard")
    out = gr.HTML()
    for trigger in (box.submit, go.click):
        trigger(render, inputs=box, outputs=out)
    for b, text in ((b1, HERO), (b2, FAILURE), (b3, WILDCARD)):
        b.click(lambda t=text: (t, render(t)), outputs=[box, out])

demo.launch(share=True, debug=False)

## Fallback: save the three demo queries as a static page

If the live link fails during the presentation, we will just open this html file.

In [ ]:
page = "".join(f"<h2>{label}: {html.escape(q)}</h2>{render(q)}<hr>"
               for label, q in (("Hero", HERO), ("Failure", FAILURE), ("Wildcard", WILDCARD)))
path = f"{DRIVE}/demo_{DATASET}_{STUD_SHORT}.html"
open(path, "w").write(f"<html><head><meta charset='utf-8'><title>GPL demo</title></head>"
                      f"<body style='font-family:sans-serif;max-width:1200px;margin:auto'>{page}</body></html>")
print("saved", path)